In [ ]:
import dataclasses
import functools as ft
import pathlib
import time

import imageio.v2 as imageio
import jax
import jax.numpy as jnp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polyscope as ps
import trimesh

import crops.src.bc as _bc
import crops.src.contact as _contact
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.mma as _mma
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz
import crops.tools.geom as _geom

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.high_res_plot_settings)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})

### Cathether case

In [ ]:
case_name = "k_009_left"
target_file = pathlib.Path(f"crops/data/catheter_obstacles/{case_name}.csv")
stl_file = pathlib.Path(f"crops/data/catheter_obstacles/{case_name}.stl")
results_dir = pathlib.Path(f"results/paper/catheter/{case_name}")
results_dir.mkdir(parents=True, exist_ok=True)
history_file = results_dir / "history.npz"  # checkpoint and final result

# Catheter tube, SI units.
outer_radius = 0.5e-3
inner_radius = 0.35e-3
shear_modulus = 3.03e5
num_elems = 120
num_fibers = 8
sheath_fraction = 0.1  # length still outside at full insertion, fraction of path
steps_per_elem = 3.75

contact_grid_spacing = 0.2e-3
contact_penalty_ratio = 1.0  # contact spring / element bending spring

### Centerline of the lumen

In [ ]:
target = pd.read_csv(target_file, comment="#")
centerline = target[["x", "y", "z"]].to_numpy()
path_length = float(target["s"].iloc[-1])
mouth = centerline[0]
entry_step = centerline[1] - centerline[0]
entry_tangent = entry_step / np.linalg.norm(entry_step)
rod_length = (1.0 + sheath_fraction) * path_length

print(f"path length {1e3 * path_length:.1f} mm, rod length {1e3 * rod_length:.1f} mm")

### Build the rod 

annular rod 

In [ ]:
mean_radius = 0.5 * (outer_radius + inner_radius)
fiber_area = np.pi * (outer_radius**2 - inner_radius**2) / num_fibers
ring_offsets = _geom.annular_fiber_offsets(
  num_elems=num_elems, num_fibers=num_fibers, semi_axis_d1=mean_radius
)

fiber_offsets = jnp.asarray(ring_offsets)
parametric_coords = jnp.linspace(0.0, rod_length, num_elems + 1)
fiber_areas = jnp.full((num_elems, num_fibers), fiber_area)
fiber_mass_density = jnp.full((num_elems, num_fibers), 1100.0)
fiber_centroid_moi = jnp.zeros((num_elems, num_fibers, 2))
least_aligned_axis = np.argmin(np.abs(entry_tangent))
seed_binormal = jnp.eye(3)[least_aligned_axis]

rod_curve = lambda arc: arc * jnp.asarray(entry_tangent)  # noqa: E731

rod = _mesher.build_rod_geometry(
  analytical_curve=rod_curve,
  parametric_coords=parametric_coords,
  fiber_offsets=fiber_offsets,
  fiber_areas=fiber_areas,
  fiber_mass_density=fiber_mass_density,
  fiber_centroid_moi=fiber_centroid_moi,
  seed_binormal=seed_binormal,
)
contact_radius = float(np.max(rod.contact_radii))
print(f"contact radius {1e3 * contact_radius:.3f} mm")

### Rod insertion (BC) and obstacle movement

In [ ]:
num_load_steps = int(np.ceil(steps_per_elem * num_elems / (1.0 + sheath_fraction)))
max_depth = path_length * 0.9
insertion_depths = max_depth * np.arange(1, num_load_steps + 1) / num_load_steps

node_arc = np.asarray(rod.arc_coords)
node_path_coords = node_arc[None, :] - rod_length + insertion_depths[:, None]
outside_nodes = [np.flatnonzero(coords <= 0.0) for coords in node_path_coords]

zero_values = [np.zeros(nodes.size) for nodes in outside_nodes]

clamped_dofs = (
  _utils.Translation.X,
  _utils.Translation.Y,
  _utils.Translation.Z,
  _utils.Rotation.THETA_X,
  _utils.Rotation.THETA_Y,
  _utils.Rotation.THETA_Z,
)
sheath = [
  _bc.DirichletBC(
    values=(dof, zero_values),
    node_targets=outside_nodes,
    name=f"sheath_{dof}",
  )
  for dof in clamped_dofs
]
bc = _bc.process_boundary_conditions(rod=rod, boundary_conditions_list=sheath)
print(f"{num_load_steps} insertion steps of {1e3 * insertion_depths[0]:.3f} mm")

In [ ]:
vessel = trimesh.load(stl_file, force="mesh")
mouth_positions = np.einsum("k,d->kd", rod_length - insertion_depths, entry_tangent)
vessel_translations = jnp.asarray(mouth_positions - mouth)
movement = _contact.ObstacleMovement.from_translations(vessel_translations)

padding = 3.0 * contact_radius
grid_extent = vessel.extents + 2.0 * padding
grid_counts = np.ceil(grid_extent / contact_grid_spacing).astype(int)
grid_shape = tuple(grid_counts)
wall_penalty_stiffness = _contact.compute_self_contact_stiffness(
  rod=rod,
  youngs_modulus=3.0 * shear_modulus,
  stiffness_ratio=contact_penalty_ratio,
)

obstacle = _contact.Obstacle.from_mesh(
  mesh=vessel,
  grid_shape=grid_shape,
  wall_penalty_stiffness=wall_penalty_stiffness,
  padding=padding,
  extrap_fill_value=10.0,
  movement=movement,
)

### Solver settings

In [ ]:
nonlinear_settings = {
  "max_iter": 100,
  "threshold": 1e-8,
  "abs_threshold": 1e-11,
  "lam_min": 0.01,
  "line_search_max_iter": 12,
  "line_search_shrink": 0.5,
  "line_search_alpha_min": 1e-6,
  "line_search_armijo_c": 1e-4,
}
linear_settings = {
  "solver": _solver.LinearSolvers.SCIPY_SPARSE,
  "rtol": 1e-10,
}

nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings=nonlinear_settings,
  linear_settings=linear_settings,
  verbose=True,
)

### Problem

In [ ]:
problem = _fe.CosseratProblem(
  rod=rod,
  bc=bc,
  nr_solver=nr_solver,
  obstacle=obstacle,
  proximal_factor=1e-1,
)
shear_moduli = jnp.full((num_elems, num_fibers), shear_modulus)
u0 = jnp.zeros(rod.num_dofs)

### Design vars and their extents

In [ ]:
max_remnant_flux = 0.2  # [T]
max_applied_flux = 2.0e-2  # [T]

remnant_shape = (num_elems,)  # one vector per element, shared by its fibers
applied_shape = (num_load_steps,)  # one field per load step
polar_extent = _utils.Extent(min=0.0, max=np.pi)
azith_extent = _utils.Extent(min=-np.pi, max=np.pi)
remnant_strength_extent = _utils.Extent(min=0.0, max=max_remnant_flux)
applied_strength_extent = _utils.Extent(min=0.0, max=max_applied_flux)

blocks = [
  ("remnant_polar", remnant_shape, polar_extent),
  ("remnant_azith", remnant_shape, azith_extent),
  ("remnant_strength", remnant_shape, remnant_strength_extent),
  ("applied_polar", applied_shape, polar_extent),
  ("applied_azith", applied_shape, azith_extent),
  ("applied_strength", applied_shape, applied_strength_extent),
]
layout = _utils.DesignLayout(blocks=blocks)

In [ ]:
field_filter_radius = 5.0e-3  # [m] of insertion depth
remnant_filter_radius = 5.0e-3  # [m] of arc length
depth_coords = insertion_depths[:, None]
field_filter = _utils.create_density_filter(
  coords=depth_coords,
  cutoff_distance=field_filter_radius,
)
elem_arc_centers = 0.5 * (node_arc[:-1] + node_arc[1:])
remnant_filter = _utils.create_density_filter(
  coords=elem_arc_centers[:, None],
  cutoff_distance=remnant_filter_radius,
)


def design_to_fluxes(design_vars: jnp.ndarray) -> tuple[jnp.ndarray, jnp.ndarray]:
  """Maps the flat design vector to remnant (E, F, 3) and applied (K, 3) flux."""
  physical = layout.to_physical(flat=design_vars)
  raw_remnant_per_elem = _utils.polar_azith_to_vector(
    polar_ang=physical["remnant_polar"],
    azith_ang=physical["remnant_azith"],
    vec_mag=physical["remnant_strength"],
  )
  # Smooth along the rod so the magnetization varies continuously with arc length.
  remnant_per_elem = remnant_filter @ raw_remnant_per_elem
  remnant_mag_flux = jnp.broadcast_to(
    remnant_per_elem[:, None, :],
    (num_elems, num_fibers, 3),
  )
  raw_applied_mag_flux = _utils.polar_azith_to_vector(
    polar_ang=physical["applied_polar"],
    azith_ang=physical["applied_azith"],
    vec_mag=physical["applied_strength"],
  )
  # Smooth over depth so the field varies continuously with insertion.
  applied_mag_flux = field_filter @ raw_applied_mag_flux
  return remnant_mag_flux, applied_mag_flux

### Init design

In [ ]:
# Axial half-strength remnant, weak field normal to t0 at azimuth -90 deg.
z_axis = np.array([0.0, 0.0, 1.0])
normal_seed = np.cross(entry_tangent, z_axis)
field_normal = normal_seed / np.linalg.norm(normal_seed)
remnant_start = jnp.broadcast_to(0.5 * max_remnant_flux * entry_tangent, (num_elems, 3))
applied_start = jnp.broadcast_to(
  0.05 * max_applied_flux * field_normal, (num_load_steps, 3)
)

remnant_polar, remnant_azith, remnant_strength = _utils.vector_to_polar_azith(
  remnant_start
)
applied_polar, applied_azith, applied_strength = _utils.vector_to_polar_azith(
  applied_start
)
start_physical = {
  "remnant_polar": remnant_polar,
  "remnant_azith": remnant_azith,
  "remnant_strength": remnant_strength,
  "applied_polar": applied_polar,
  "applied_azith": applied_azith,
  "applied_strength": applied_strength,
}
initial_design = layout.to_flat(start_physical)

### Optimization objective

In [ ]:
def compute_net_wall_force(forces: jnp.ndarray) -> jnp.ndarray:
  """Sums the wall force magnitude over all nodes and load steps."""
  squared = jnp.einsum("knd,knd->kn", forces, forces)
  return jnp.sum(_utils.safe_sqrt(x=squared))


def compute_objective(
  design_vars: jnp.ndarray,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
  """Returns the net wall force (1,) [N], its gradient (n, 1), and disp (K, dofs)."""

  def objective_fn(design_vars: jnp.ndarray) -> tuple[jnp.ndarray, jnp.ndarray]:
    remnant_mag_flux, applied_mag_flux = design_to_fluxes(design_vars=design_vars)
    _, displacement_history = problem.solve(
      u0=u0,
      shear_moduli=shear_moduli,
      remnant_mag_flux=remnant_mag_flux,
      applied_mag_flux=applied_mag_flux,
    )
    forces, _ = _contact.compute_wall_forces_and_moments(
      rod=rod,
      obstacle=obstacle,
      displacements=displacement_history,
    )
    return compute_net_wall_force(forces=forces), displacement_history

  (objective, displacement_history), grad_objective = jax.value_and_grad(
    objective_fn, has_aux=True
  )(design_vars)
  return objective.reshape(1), grad_objective.reshape(-1, 1), displacement_history


def compute_wall_force(displacement_history: jnp.ndarray) -> np.ndarray:
  """Returns the wall force (K,) [N], nodal contact force magnitudes summed per step."""
  forces, _ = _contact.compute_wall_forces_and_moments(
    rod=rod,
    obstacle=obstacle,
    displacements=jnp.asarray(displacement_history),
  )
  return np.linalg.norm(forces, axis=-1).sum(axis=1)

### Plotting codes

In [ ]:
unmagnetized_style = {"color": "tab:red", "linestyle": "--", "label": "Unmagnetized"}
optimized_style = {"color": "tab:green", "linestyle": "-", "label": "Optimized"}
run_styles = {"unmagnetized": unmagnetized_style, "optimized": optimized_style}
top_legend = {"loc": "lower center", "bbox_to_anchor": (0.5, 1.02), "frameon": True}
depth_mm = 1e3 * insertion_depths

entry_frame = np.asarray(rod.directors_0[0])  # rows d1, d2, d3 = t
flux_styles = (  # (row of entry_frame, color, label)
  (2, "red", r"$\mathbf{t}$"),
  (0, "olive", r"$\mathbf{d}_1$"),
  (1, "navy", r"$\mathbf{d}_2$"),
)
# The applied flux is a lab-frame field, so its components are along the fixed rest
# directors at the entry, not the rod's current ones.
applied_flux_labels = (r"$\mathbf{t}_0$", r"$\mathbf{d}_{1,0}$", r"$\mathbf{d}_{2,0}$")

# 3D contact renders.
contact_threshold = 0.05  # of the reference force, weaker contacts get no sphere
max_sphere_radius = 7.0 * contact_radius  # sphere radius at the reference force
node_radii = _viz.elem_to_node_values(rod.contact_radii)
gizmo_radius = 0.08 * float(np.ptp(centerline, axis=0).max())


def plot_convergence(
  objective_history: np.ndarray, unmagnetized_objective: float
) -> plt.Figure:
  """Plots the objective, relative to the unmagnetized rod's, against epoch."""
  relative_history = np.asarray(objective_history) / unmagnetized_objective
  fig, ax = plt.subplots(figsize=(5.0, 3.5))
  ax.axhline(1.0, **unmagnetized_style)
  ax.plot(relative_history, **optimized_style)
  ax.set_xlabel("Epoch")
  ax.set_ylabel(r"Objective $J / J_{\mathrm{unmag}}$")
  ax.legend(ncol=2, **top_legend)
  return fig


def plot_wall_force(
  unmagnetized_wall_force: np.ndarray, optimized_wall_force: np.ndarray
) -> plt.Figure:
  """Plots the wall force (K,) [N] of both rods against insertion depth, in mN."""
  fig, ax = plt.subplots(figsize=(5.0, 3.5))
  ax.plot(depth_mm, 1e3 * unmagnetized_wall_force, **unmagnetized_style)
  ax.plot(depth_mm, 1e3 * optimized_wall_force, **optimized_style)
  ax.set_xlabel(r"Insertion depth $d$ [mm]")
  ax.set_ylabel("Wall force [mN]")
  ax.legend(ncol=2, **top_legend)
  return fig


def plot_flux_components(
  coords: np.ndarray, flux: np.ndarray, labels: tuple[str, ...] | None = None
) -> tuple[plt.Figure, plt.Axes]:
  """Plots the entry-frame components and magnitude of a (n, 3) flux [T] in mT.

  `labels` name the components in the order of `flux_styles`, whose material-frame
  labels are the default.
  """
  components = 1e3 * flux @ entry_frame.T
  if labels is None:
    labels = [label for _, _, label in flux_styles]
  fig, ax = plt.subplots(figsize=(5.0, 3.5))
  ax.axhline(0.0, color="0.75", linewidth=0.8)
  for (row, color, _), label in zip(flux_styles, labels):
    ax.plot(coords, components[:, row], color=color, label=label)
  magnitude = np.linalg.norm(components, axis=1)
  ax.plot(coords, magnitude, color="k", linestyle="--", label=r"$|\mathbf{B}|$")
  ax.legend(ncol=4, columnspacing=1.0, handlelength=1.5, **top_legend)
  return fig, ax


def setup_contact_scene(window_size: tuple[int, int],
                        plot_mag_gizmo: bool = False) -> tuple[slice, slice]:
  """Registers the vessel, field gizmo, and rod in Polyscope, returns the crop box.

  The vessel stays put in its own body frame and each frame moves the rod into it.
  The camera looks at the centerline face-on. The crop box is measured on the vessel
  and gizmo alone, so the rod's sheathed part outside the vessel is cropped away.
  The scene extents are then frozen, so the gizmo arrows keep their thickness as the
  rod moves. The renders are twice `window_size` on a Retina display.
  """
  _viz.init_scene_ps(window_size=window_size, ssaa_factor=2, transparency_mode="pretty")
  ps.remove_all_structures()
  ps.set_automatically_compute_scene_extents(True)
  _viz.plot_obstacle_ps(
    obstacle=dataclasses.replace(obstacle, movement=None),
    name="vessel",
    transparency=0.15,
  )
  _viz.look_at_best_fit_plane_ps(points=centerline)
  if plot_mag_gizmo:
    _viz.plot_direction_gizmo_ps(
      center=_viz.top_left_anchor_ps(points=vessel.vertices, radius=gizmo_radius),
      radius=gizmo_radius,
      frame=entry_frame[[row for row, _, _ in flux_styles]],
      frame_colors=tuple(mpl.colors.to_rgb(color) for _, color, _ in flux_styles),
      name="field",
    )
  crop_box = _viz.content_crop_box_ps()
  ps.set_automatically_compute_scene_extents(False)
  _viz.plot_rod_centerline_ps(
    node_coords=np.zeros((rod.num_nodes, 3)),
    name="rod",
    base_color=(0.3, 0.3, 0.3),
    node_radii=node_radii,
  )
  return crop_box


def render_contact_frame(
  run: dict[str, np.ndarray],
  step: int,
  crop_box: tuple[slice, slice],
  reference_force: float,
  plot_mag_gizmo: bool = False,
) -> np.ndarray:
  """Poses the rod, contact spheres and field arrow at one load step, returns RGB."""
  obj_trans, obj_rot_mat = obstacle.pose_at_step(step_idx=step)
  to_vessel = ft.partial(
    _utils.affine_transform,
    rotation_matrix=obj_rot_mat,
    direction="GLOBAL_TO_LOCAL",
  )
  node_coords = np.asarray(
    to_vessel(coords=run["node_coords"][step], translation=obj_trans)
  )
  # Forces and fields are directions, so they rotate but do not translate.
  forces = np.asarray(to_vessel(coords=run["forces"][step], translation=np.zeros(3)))
  field = np.asarray(
    to_vessel(coords=run["applied_flux"][step], translation=np.zeros(3))
  )
  _viz.update_rod_centerline_ps(node_coords=node_coords, name="rod")
  _viz.plot_contact_spheres_ps(
    node_coords=node_coords,
    forces=forces,
    node_radii=node_radii,
    reference_force=reference_force,
    max_radius=max_sphere_radius,
    threshold=contact_threshold,
  )
  if plot_mag_gizmo:
    _viz.update_direction_gizmo_ps(
      direction=field, length=2.0 * gizmo_radius, name="field"
    )
  return _viz.screenshot_ps(crop_box=crop_box)


def add_contact_colorbar(
  fig: plt.Figure, axes: np.ndarray, reference_force: float
) -> None:
  """Adds the contact force colorbar, from zero to the reference force, in mN."""
  mappable = mpl.cm.ScalarMappable(
    norm=mpl.colors.Normalize(vmin=0.0, vmax=1e3 * reference_force),
    cmap=_viz.contact_force_cmap,
  )
  fig.colorbar(mappable, ax=axes, label="Contact force [mN]", shrink=0.8, pad=0.02)


def plot_contact_snapshots(
  runs: dict[str, dict[str, np.ndarray]],
  steps: list[int],
  reference_force: float,
  render_dir: pathlib.Path,
  window_size: tuple[int, int] = (1800, 1800),
) -> plt.Figure:
  """Renders every run at the given load steps as a grid, one row per run.

  Each render is also written at full resolution to
  `render_dir/<run>_contact_<step>.png`, for laying out the figure by hand. The
  default window gives renders about 3000 px on the longer side, so save the grid
  with `dpi` of about 800 to keep that resolution in a PDF.
  """
  crop_box = setup_contact_scene(window_size=window_size)
  fig, axes = plt.subplots(
    len(runs),
    len(steps),
    figsize=(3.2 * len(steps), 6.6),
    squeeze=False,
    layout="constrained",
  )
  for row, (run_name, run) in enumerate(runs.items()):
    for col, step in enumerate(steps):
      image = render_contact_frame(
        run=run, step=step, crop_box=crop_box, reference_force=reference_force
      )
      imageio.imwrite(render_dir / f"{run_name}_contact_{step:04d}.png", image)
      ax = axes[row, col]
      ax.imshow(image)
      ax.set_xticks([])
      ax.set_yticks([])
      ax.spines[:].set_visible(False)
      ax.set_xlabel(f"wall force {1e3 * run['wall_force'][step]:.2f} mN", fontsize=13)
      if row == 0:
        ax.set_title(rf"$d$ = {depth_mm[step]:.0f} mm")
    style = run_styles[run_name]
    axes[row, 0].set_ylabel(style["label"], color=style["color"])
  add_contact_colorbar(fig=fig, axes=axes, reference_force=reference_force)
  return fig


def save_contact_gif(
  runs: dict[str, dict[str, np.ndarray]],
  steps: list[int],
  reference_force: float,
  file_path: pathlib.Path,
  window_size: tuple[int, int] = (1000, 1000),
  dpi: int = 150,
  frame_ms: int = 80,
  hold_ms: int = 1500,
) -> None:
  """Animates the runs side by side over the given load steps as a looping GIF.

  The GIF is `8.5 dpi` px wide. Each frame shows `frame_ms` and the last one is held
  for `hold_ms`.
  """
  crop_box = setup_contact_scene(window_size=window_size)
  fig, axes = plt.subplots(1, len(runs), figsize=(8.5, 4.2), dpi=dpi)
  panels = {}
  for ax, (run_name, run) in zip(axes, runs.items()):
    style = run_styles[run_name]
    ax.set_title(style["label"], color=style["color"])
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines[:].set_visible(False)
    image = render_contact_frame(
      run=run, step=steps[0], crop_box=crop_box, reference_force=reference_force
    )
    force_text = ax.text(
      0.97, 0.97, "", transform=ax.transAxes, ha="right", va="top", fontsize=14
    )
    panels[run_name] = (ax.imshow(image), force_text)
  depth_text = fig.text(0.45, 0.06, "", ha="center")
  add_contact_colorbar(fig=fig, axes=axes, reference_force=reference_force)

  frames = []
  for step in steps:
    for run_name, run in runs.items():
      image_artist, force_text = panels[run_name]
      image_artist.set_data(
        render_contact_frame(
          run=run, step=step, crop_box=crop_box, reference_force=reference_force
        )
      )
      force_text.set_text(f"wall force\n{1e3 * run['wall_force'][step]:.2f} mN")
    depth_text.set_text(rf"$d$ = {depth_mm[step]:.1f} mm")
    fig.canvas.draw()
    frames.append(np.asarray(fig.canvas.buffer_rgba())[..., :3].copy())
  plt.close(fig)
  durations = [frame_ms] * (len(frames) - 1) + [hold_ms]
  imageio.mimsave(file_path, frames, duration=durations, loop=0)

### Optimization
The unmagnetized baseline, the same rod with no remnant or applied field, is solved once per run inside `run_optimization` and kept in the history, so post-processing never re-solves.

In [ ]:
def run_optimization(
  history_file: pathlib.Path,
  resume: bool = False,
  max_epochs: int = 150,
  move_limit: float = 6e-3,
  normalize_epochs: tuple[int, ...] = (0, 10),
  plot_interval: int = 3,
) -> dict[str, np.ndarray]:
  """Minimizes the net wall force with MMA and writes the history after every epoch.

  The history file is both the checkpoint and the final result. It is replaced
  atomically after every epoch, so a run stopped at any point leaves a complete file
  that `resume` continues from and the post-processing reads. Only the last evaluated
  epoch's displacement history is kept, next to that of the unmagnetized rod, so the
  file stays small while every result figure is drawn without a solve.

  Algorithm.
    1. Compile the objective. Without `resume`, start MMA from `initial_design` and
       solve the unmagnetized rod once, the baseline of every comparison. With
       `resume`, restore the MMA state, objective scale, history, and baseline from
       `history_file`.
    2. Evaluate the net wall force `J` and its gradient. Stop, and write the state to
       `debug.npz` beside the history, if either is non-finite.
    3. At each epoch in `normalize_epochs`, set the scale to the current `J`. Pass
       `J` and its gradient divided by the scale to MMA, so the move limit and the
       tolerances act on a quantity of order one.
    4. Update the design with one MMA step against an inactive dummy constraint,
       timing the whole epoch.
    5. Record the epoch, `J`, the epoch time, and the evaluated design, redraw the
       convergence and the wall force every `plot_interval` epochs, and write the
       history.
    6. Stop when MMA converges or at `max_epochs`.

  Args:
    history_file: Path of the `.npz` history, written after every epoch.
    resume: Whether to continue from `history_file` instead of starting afresh.
    max_epochs: Maximum number of MMA epochs, counted across resumes.
    move_limit: MMA move limit per epoch, in `(0, 1]`.
    normalize_epochs: Epochs at which the objective scale is reset.
    plot_interval: Number of epochs between monitoring plots, positive.

  Returns:
    The contents of `history_file` as a dictionary of arrays. `epoch`, `objective`
      [N], and `time_epoch` [s] have shape `(num_epochs,)`. `design` has shape
      `(num_epochs, num_design_var)` and holds the design evaluated at each epoch.
      `disp_history` and `unmagnetized_disp_history` have shape
      `(num_load_steps, num_dofs)` [m, rad], of the last evaluated design and of the
      unmagnetized rod. `mma_state` and `objective_scale` are what `resume` needs.
  """
  objective_and_grad = jax.jit(compute_objective)
  num_design_var = layout.num_design_var
  mma_params = _mma.MMAParams(
    max_iter=max_epochs,
    kkt_tol=1e-8,
    step_tol=1e-8,
    move_limit=move_limit,
    num_design_var=num_design_var,
    num_cons=1,
    lower_bound=np.zeros((num_design_var, 1)),
    upper_bound=np.ones((num_design_var, 1)),
  )
  # No constraints, so MMA gets one inactive dummy.
  dummy_cons = np.full((1, 1), -1.0)
  dummy_grad_cons = np.zeros((1, num_design_var))
  keys = ("epoch", "objective", "time_epoch", "design")

  if resume:
    saved = dict(np.load(history_file))
    mma_state = _mma.MMAState.from_array(
      state_array=saved["mma_state"],
      num_design_var=num_design_var,
    )
    # MMA also flags convergence on reaching its epoch cap, which must not stop a
    # resume with a larger `max_epochs`, so keep the flag only for a met tolerance.
    mma_state.is_converged = bool(
      mma_state.kkt_norm < mma_params.kkt_tol
      or mma_state.change_design_var < mma_params.step_tol
    )
    objective_scale = float(saved["objective_scale"])
    history = {key: list(saved[key]) for key in keys}
    unmagnetized_disp_history = saved["unmagnetized_disp_history"]
  else:
    saved = {}
    mma_state = _mma.init_mma(
      init_design_var=np.asarray(initial_design).reshape(-1, 1),
      mma_params=mma_params,
    )
    objective_scale = 1.0
    history = {key: [] for key in keys}
    # Baseline for every comparison: the same rod with no remnant or applied field.
    _, unmagnetized_disp_history = problem.solve(u0=u0, shear_moduli=shear_moduli)
    unmagnetized_disp_history = np.asarray(unmagnetized_disp_history)
  unmagnetized_wall_force = compute_wall_force(
    displacement_history=unmagnetized_disp_history
  )
  temp_file = history_file.with_name(f"{history_file.stem}_tmp.npz")

  while mma_state.epoch < max_epochs and not mma_state.is_converged:
    epoch = mma_state.epoch
    design_vars = jnp.asarray(mma_state.x).reshape(-1)

    epoch_start = time.perf_counter()
    objective, grad_objective, disp_history = objective_and_grad(design_vars)

    if epoch in normalize_epochs:
      objective_scale = float(objective[0])
    
    normalized_objective = np.asarray(objective, dtype=np.float64) / objective_scale
    normalized_grad = np.asarray(grad_objective, dtype=np.float64) / objective_scale
    
    mma_state = _mma.update_mma(
      mma_state=mma_state,
      mma_params=mma_params,
      obj=normalized_objective,
      grad_obj=normalized_grad,
      cons=dummy_cons,
      grad_cons=dummy_grad_cons,
    )
    time_epoch = time.perf_counter() - epoch_start

    history["epoch"].append(epoch)
    history["objective"].append(float(objective[0]))
    history["time_epoch"].append(time_epoch)
    history["design"].append(np.asarray(design_vars))
    print(
      f"epoch {epoch:3d}  J {float(normalized_objective[0]):.4f}  {time_epoch:.1f} s"
    )

    if epoch % plot_interval == 0:
      plt.close("all")
      plot_convergence(
        objective_history=history["objective"],
        unmagnetized_objective=float(unmagnetized_wall_force.sum()),
      )
      plot_wall_force(
        unmagnetized_wall_force=unmagnetized_wall_force,
        optimized_wall_force=compute_wall_force(displacement_history=disp_history),
      )
      plt.show()
      plt.pause(1e-6)

    # The saved MMA state already holds the next design, so a resume continues from it.
    saved = {
      **{key: np.asarray(value) for key, value in history.items()},
      "disp_history": np.asarray(disp_history),
      "unmagnetized_disp_history": unmagnetized_disp_history,
      "mma_state": mma_state.to_array(),
      "objective_scale": objective_scale,
    }
    np.savez(temp_file, **saved)
    temp_file.replace(history_file)
  return saved

### Run

In [ ]:
run = False
resume = False  # continue from history_file, e.g. after stopping the kernel

if run:
  run_optimization(history_file=history_file, resume=resume)

### Results
Everything below reads `history.npz`, nothing is re-solved. *Unmagnetized* is the baseline rod with no remnant or applied field and *optimized* is the last evaluated epoch.

The **wall force** at insertion depth $d$ is the total contact force the vessel exerts on the catheter, $W(d) = \sum_n |\mathbf{f}_n(d)|$, summed over the rod's nodes. The objective is $J = \sum_k W(d_k)$ over all insertion steps.

In [ ]:
history = np.load(history_file)
remnant_mag_flux, applied_mag_flux = design_to_fluxes(
  design_vars=jnp.asarray(history["design"][-1])
)
remnant_flux = np.asarray(remnant_mag_flux[:, 0])
applied_flux = np.asarray(applied_mag_flux)

run_inputs = {  # (displacement history, applied flux)
  "unmagnetized": (history["unmagnetized_disp_history"], np.zeros_like(applied_flux)),
  "optimized": (history["disp_history"], applied_flux),
}
runs = {}
for run_name, (disp_history, run_applied_flux) in run_inputs.items():
  disp_history = jnp.asarray(disp_history)
  node_coords, _, _ = _mesher.extract_coords_and_director_from_deformation(
    rod=rod,
    u=disp_history,
  )
  forces, _ = _contact.compute_wall_forces_and_moments(
    rod=rod,
    obstacle=obstacle,
    displacements=disp_history,
  )
  runs[run_name] = {
    "node_coords": np.asarray(node_coords),
    "forces": np.asarray(forces),
    "wall_force": np.linalg.norm(forces, axis=-1).sum(axis=1),
    "applied_flux": run_applied_flux,
  }

summary = pd.DataFrame(
  {
    run_name: {
      "mean [mN]": 1e3 * run["wall_force"].mean(),
      "peak [mN]": 1e3 * run["wall_force"].max(),
    }
    for run_name, run in runs.items()
  }
).T
summary.loc["change [%]"] = 100.0 * (
  summary.loc["optimized"] / summary.loc["unmagnetized"] - 1.0
)
print(f"{len(history['epoch'])} epochs")
print(summary.round(3))

In [ ]:
fig = plot_convergence(
  objective_history=history["objective"],
  unmagnetized_objective=float(runs["unmagnetized"]["wall_force"].sum()),
)
fig.savefig(results_dir / "convergence.pdf")

fig = plot_wall_force(
  unmagnetized_wall_force=runs["unmagnetized"]["wall_force"],
  optimized_wall_force=runs["optimized"]["wall_force"],
)
fig.savefig(results_dir / "wall_force.pdf")

### Magnetization of the optimized design
Components in the entry frame: tangent $\mathbf{t}$ (red), $\mathbf{d}_1$ (olive), $\mathbf{d}_2$ (navy), and the magnitude (black). The rest rod is straight along the entry, so this frame is the material frame of the remnant flux and a fixed lab frame for the applied flux. The remnant is plotted against distance from the tip, so the element at distance $x$ enters the vessel at insertion depth $d = x$; the part of the rod that never enters is not shown.

In [ ]:
# Elements further from the tip than the deepest insertion never enter the vessel.
tip_distance = rod_length - elem_arc_centers
inserted = tip_distance <= max_depth

fig, ax = plot_flux_components(
  coords=1e3 * tip_distance[inserted], flux=remnant_flux[inserted]
)
ax.set_xlabel("Distance from tip [mm]")
ax.set_ylabel(r"Remnant flux $\mathbf{B}^r$ [mT]")
fig.savefig(results_dir / "remnant_flux.pdf")

fig, ax = plot_flux_components(
  coords=depth_mm, flux=applied_flux, labels=applied_flux_labels
)
ax.set_xlabel(r"Insertion depth $d$ [mm]")
ax.set_ylabel(r"Applied flux $\mathbf{B}^a$ [mT]")
fig.savefig(results_dir / "applied_flux.pdf")

### Contact in 3D
A translucent sphere marks every node whose contact force exceeds `contact_threshold` of the reference force, placed on the rod surface facing the wall. Both color and radius ($\propto \sqrt{f}$) encode the force on one scale for both rods, set by the peak nodal contact force of the unmagnetized rod. The gizmo beside the vessel shows the applied field direction (black arrow) against the entry frame $\mathbf{t}$ (red), $\mathbf{d}_1$ (olive), $\mathbf{d}_2$ (navy); the unmagnetized rod has no arrow. Each still is also saved on its own at full resolution as `<run>_contact_<step>.png`.

In [ ]:
# One scale for both rods, the peak nodal contact force of the unmagnetized rod.
reference_force = float(np.linalg.norm(runs["unmagnetized"]["forces"], axis=-1).max())
print(f"reference contact force {1e3 * reference_force:.3f} mN")

snapshot_fractions = (0.25, 0.5, 0.75, 1.0)  # of the insertion
snapshot_steps = [round(frac * (num_load_steps - 1)) for frac in snapshot_fractions]
fig = plot_contact_snapshots(
  runs=runs,
  steps=snapshot_steps,
  reference_force=reference_force,
  render_dir=results_dir,
)
fig.savefig(results_dir / "contact_snapshots.pdf", dpi=800)

In [ ]:
frame_interval = 6  # load steps between gif frames
gif_steps = [*range(0, num_load_steps - 1, frame_interval), num_load_steps - 1]
save_contact_gif(
  runs=runs,
  steps=gif_steps,
  reference_force=reference_force,
  file_path=results_dir / "contact_insertion.gif",
)